# 02 — Majority-class baseline and reproducible data split

**Objective.** Quantify the minimum predictive performance to beat on an imbalanced machine-failure task, and establish a consistent 80/20 stratified split for subsequent model comparisons.

**Workflow.** Load and check the approved schema → reproduce the seeded split → fit an all-normal baseline → examine its confusion matrix and class-specific metrics.

This notebook is self-contained; it does not require any other notebook or Python helper file.

## 1. Load the exact input schema

We use only the six approved inputs and the original binary target. No fitting or learned preprocessing occurs in this step.

In [9]:
from IPython.display import display
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)

from pathlib import Path
from sklearn.model_selection import train_test_split

# Exact data schema used throughout the original experiments.
RANDOM_STATE = 42
TARGET = "Machine failure"
CATEGORICAL_FEATURES = ["Type"]
NUMERIC_FEATURES = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]",
]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# Accept execution from the repository root or its notebooks/ directory.
# No extra .py helper file is needed.
DATA_PATH = Path("data/raw/ai4i2020.csv")
if not DATA_PATH.is_file():
    DATA_PATH = Path("../data/raw/ai4i2020.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Cannot locate data/raw/ai4i2020.csv. Open the project root "
        "or its notebooks/ folder in VS Code and keep the original CSV in data/raw/. "
        f"Current working directory: {Path.cwd()}"
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")


Dataset loaded: 10,000 rows × 14 columns


In [10]:
# These six inputs are available at prediction time; failure-mode flags and IDs
# are deliberately excluded because they could leak the target or identify rows.
required_columns = FEATURES + [TARGET]
missing_columns = [name for name in required_columns if name not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
if df[required_columns].isna().any().any():
    raise ValueError("Missing values found in the selected modelling columns.")

X = df[FEATURES].copy()
y = df[TARGET].copy()
print(f"Predictor matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

Predictor matrix shape: (10000, 6)
Target vector shape: (10000,)


## 2. Establish the reproducible data split

The split uses `test_size=0.20`, `stratify=y` and `random_state=42`. Stratification preserves the rare-failure proportion and makes later comparisons on matching rows possible. All model training and cross-validation are based on the 8,000-row development set.

**Evaluation-history limitation.** The 2,000-row test subset was examined during the original experimentation, including this baseline; it is reproducibly separated from training, but should not be represented as an untouched, fully independent confirmation set.

In [11]:
# Use precisely the same stratified, seeded 80/20 row split in every notebook.
# Stratification maintains approximately 3.39% failures in each subset.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y,
)

split_summary = pd.DataFrame({
    "Observations": [len(y_train), len(y_test)],
    "Failures": [int(y_train.sum()), int(y_test.sum())],
    "Failure Rate (%)": [100 * y_train.mean(), 100 * y_test.mean()],
}, index=["Training", "Test"])
display(split_summary.round(2))

,Observations,Failures,Failure Rate (%)
Training,8000,271,3.39
Test,2000,68,3.40


## 3. Evaluate the majority-class baseline

`DummyClassifier(strategy="most_frequent")` learns only the most frequent development label and predicts that class for every test record. The confusion matrix has actual classes in rows and predicted classes in columns (`TN`, `FP`, `FN`, `TP`). Recall is `TP/(TP+FN)` and precision is `TP/(TP+FP)`; F1 summarises their balance.

In [12]:
# This intentionally trivial baseline always predicts the training majority class.
baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

# Accuracy alone can disguise a model that misses every failure.
baseline_metrics = pd.DataFrame({"Score": {
    "Accuracy": accuracy_score(y_test, y_pred_baseline),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred_baseline),
    "Precision": precision_score(y_test, y_pred_baseline, zero_division=0),
    "Recall": recall_score(y_test, y_pred_baseline, zero_division=0),
    "F1-score": f1_score(y_test, y_pred_baseline, zero_division=0),
}})
print("Majority-class baseline performance:")
display(baseline_metrics.round(4))

# For labels [0, 1]: [[TN, FP], [FN, TP]].
cm = confusion_matrix(y_test, y_pred_baseline, labels=[0, 1])
print("\nConfusion matrix (actual rows, predicted columns):")
display(pd.DataFrame(cm,
    index=["Actual normal", "Actual failure"],
    columns=["Predicted normal", "Predicted failure"],
))

Majority-class baseline performance:


,Score
Accuracy,0.966
Balanced Accuracy,0.500
Precision,0.000
Recall,0.000
F1-score,0.000



Confusion matrix (actual rows, predicted columns):


,Predicted normal,Predicted failure
Actual normal,1932,0
Actual failure,68,0


## Baseline Findings and Engineering Implications

The dataset was divided into 8,000 development and 2,000 test observations using a stratified split to maintain similar failure rates.

The majority-class baseline predicted every observation as normal. Although it achieved 96.60% test accuracy, it missed all 68 recorded failures, giving a recall and F1-score of zero.

This demonstrates why accuracy alone is misleading when failures are rare. A useful maintenance classifier must detect failures while keeping false alarms manageable.

**Conclusion:** The baseline provides a minimum reference for later models. Improving failure detection, rather than accuracy alone, is the main objective.